# Dynamics of S6b and S6c

As step 7, for the two models of the acquisition round: S6b (two-sided acquisition) and S6c (control).

Input: `step9_bundle.zip` with `batched_md.py`, `li3ocl_S6b.model` and `li3ocl_S6c.model` from this archive. Runtime: GPU. Output: one `hops_*.npz` per job in the Google Drive folder; running all cells again resumes an interrupted session.

In [ ]:
!pip -q install mace-torch ase
import torch, time, json, os, numpy as np
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
JOBS = [(s, T) for T in (1000.0, 900.0, 800.0) for s in ('S6b', 'S6c')]
HOPS_TARGET, NS_MAX, B, CHUNK = 400, 0.10, 64, 2500
OUT = '.'
try:
    from google.colab import drive; drive.mount('/content/drive'); OUT = '/content/drive/MyDrive/li3ocl_step9'; os.makedirs(OUT, exist_ok=True)
except Exception as err: print('no Drive, working locally:', err)
if not os.path.exists('batched_md.py'):
    from google.colab import files
    up = files.upload()            # choose step9_bundle.zip
    os.system('unzip -o -q step9_bundle.zip')
print(OUT, sorted(os.listdir('.')))

In [ ]:
from ase import Atoms
from mace.calculators import MACECalculator
from batched_md import BatchedMD, HopCounter
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; A = 3.926
unit = Atoms('ClOLi3', scaled_positions=[(0,0,0), (.5,.5,.5), (.5,.5,0), (.5,0,.5), (0,.5,.5)], cell=[A]*3, pbc=True)
s = unit.repeat((3,3,3)); li_all = [i for i, z in enumerate(s.get_chemical_symbols()) if z == 'Li']; SITES = s.positions[li_all].copy(); del s[li_all[0]]
sym = s.get_chemical_symbols(); LI = np.array([i for i, z in enumerate(sym) if z == 'Li']); FW = np.array([i for i, z in enumerate(sym) if z != 'Li'])
MODELS = {}
def get_model(name):
    if name not in MODELS: MODELS[name] = MACECalculator(model_paths=f'li3ocl_{name}.model', device=DEV, default_dtype='float32').models[0]
    return MODELS[name]
def counter(eng): return HopCounter(eng, LI, SITES, window=25, framework_idx=FW, framework_ref=s.positions[FW].mean(0))
def start(T, n, seed):
    rng = np.random.default_rng(seed); return np.stack([s.positions + rng.normal(0, 0.05, s.positions.shape) for _ in range(n)])
# speed and time estimate
eng = BatchedMD(get_model('S6b'), s.numbers, s.cell.lengths(), start(1000.0, B, 0), 1000.0, device=DEV, seed=0); eng.run(20)
if DEV == 'cuda': torch.cuda.synchronize()
t0 = time.time(); eng.run(50)
if DEV == 'cuda': torch.cuda.synchronize()
ms = (time.time() - t0) / 50 * 1e3; nsph = 3600 / (ms * 1e-3) * 2e-6 * B; RATE = {1000.0: 134, 900.0: 69, 800.0: 34, 700.0: 12}
est = sum(min(HOPS_TARGET / RATE[T], NS_MAX * B) / nsph for _, T in JOBS)
print(f'S6b: {ms:.0f} ms per step for {B} replicas = {nsph:.1f} ns per hour.  {len(JOBS)} jobs, roughly {est:.0f} hours in total if the students hop like the teacher.')

In [ ]:
SUMMARY = {}; t_start = time.time()
for name, T in JOBS:
    tag = f'{name}_{int(T)}K'; fn, ck = f'{OUT}/hops_{tag}.npz', f'{OUT}/ckpt_{tag}.pt'
    if os.path.exists(fn):
        d = np.load(fn); SUMMARY[tag] = json.loads(str(d['summary'])); print(tag, 'done:', SUMMARY[tag]); continue
    seed = int(T) + 17 * sum(map(ord, name)); eng = BatchedMD(get_model(name), s.numbers, s.cell.lengths(), start(T, B, seed), T, device=DEV, seed=seed)
    dead = np.full(B, -1, dtype=np.int64)                          # step at which a replica was censored (-1: alive)
    if os.path.exists(ck):
        st = torch.load(ck, weights_only=False); eng.x, eng.v, eng.step_count = st['x'].to(DEV), st['v'].to(DEV), st['step']; eng._build_neighbours(); eng.e, eng.f = eng.energy_forces(); hc = counter(eng)
        hc.assign, hc.hops, hc.x0 = st['assign'].to(DEV), st['hops'].to(DEV), st['x0'].to(DEV); hc.msd, hc.events, hc.frames, hc.saved_to, hc.rec_until, hc.buf, dead = st['msd'], st['events'], st['frames'], st['saved_to'], st['rec_until'], st['buf'], st['dead']
        print(f'{tag}: resumed at {eng.step_count * 2e-6:.3f} ns per replica')
    else:
        eng.run(2500); eng.step_count = 0; hc = counter(eng)
    while True:
        ev = np.array(hc.events).reshape(-1, 5); good = sum(1 for e in ev if dead[e[1]] < 0 or e[0] <= dead[e[1]]); ns = eng.step_count * 2e-6
        if good >= HOPS_TARGET or ns >= NS_MAX: break
        n0 = eng.step_count; eng.run(CHUNK, callback=hc, callback_every=5); Tr = eng.temperature(); bad = np.where(((~np.isfinite(Tr)) | (Tr > 2 * T)) & (dead < 0))[0]
        for b in bad:                                              # censor at the last good chunk, restart the replica from the lattice so that it cannot disturb the batch
            dead[b] = n0; eng.x[b] = torch.as_tensor(s.positions, dtype=eng.x.dtype, device=eng.x.device); eng.v[b] = 0
        if len(bad): eng._build_neighbours(); eng.e, eng.f = eng.energy_forces()
        torch.save(dict(x=eng.x.cpu(), v=eng.v.cpu(), step=eng.step_count, assign=hc.assign.cpu(), hops=hc.hops.cpu(), x0=hc.x0.cpu(), msd=hc.msd, events=hc.events, frames=hc.frames,
                        saved_to=hc.saved_to, rec_until=hc.rec_until, buf=hc.buf, dead=dead), ck + '.tmp'); os.replace(ck + '.tmp', ck)
        alive_T = Tr[dead < 0].mean() if (dead < 0).any() else float('nan')
        print(f'{tag}: {eng.step_count * 2e-6:.3f} ns/replica, good hops {good}, censored replicas {(dead >= 0).sum()}, T = {alive_T:.0f} K, {(time.time() - t_start) / 3600:.2f} h', flush=True)
    ev = np.array(hc.events).reshape(-1, 5); keep_e = np.array([dead[e[1]] < 0 or e[0] <= dead[e[1]] for e in ev], bool) if len(ev) else np.zeros(0, bool)
    fr = hc.frames; keep_f = [dead[f[1]] < 0 or f[0] <= dead[f[1]] for f in fr]; fr = [f for f, k in zip(fr, keep_f) if k]
    time_ns = float(sum((eng.step_count if dead[b] < 0 else dead[b]) for b in range(B)) * 2e-6); hp = np.bincount(ev[keep_e][:, 1], minlength=B) if keep_e.any() else np.zeros(B, int)
    summ = dict(student=name, T=T, ns_total=round(time_ns, 3), hops=int(keep_e.sum()), hops_per_ns=round(float(keep_e.sum() / max(time_ns, 1e-9)), 1), censored=int((dead >= 0).sum()),
                var_over_mean=round(float(hp[dead < 0].var() / max(hp[dead < 0].mean(), 1e-9)), 2) if (dead < 0).any() else None)
    np.savez_compressed(fn, events=ev[keep_e] if len(ev) else ev, hops=hp, dead=dead, frame_step=np.array([f[0] for f in fr]), frame_replica=np.array([f[1] for f in fr]),
                        frames=np.array([f[2] for f in fr]) if fr else np.zeros((0, len(s), 3), np.float32), sites=SITES, numbers=s.numbers, cell=s.cell.lengths(),
                        ns_per_replica=eng.step_count * 2e-6, ns_total=time_ns, T=T, summary=json.dumps(summ))
    os.remove(ck); SUMMARY[tag] = summ; print('FINISHED', summ, flush=True)

In [ ]:
print('=========== SUMMARY ==========='); print(json.dumps(SUMMARY, indent=1)); print('==============================')